# Image Cluster Notebook
This notebook is designed to create labels for NAC images through K-Means clustering. It does 2 main passes: the initial K-means pass where the algorithm infers pixel grouping from their pixel values. A second pass is then done where the user must select the cluster values that belong to class 0 (no crater) and 1 (crater). These are then submitted to the display function and shown at the end of the notebook. 

## Imports

In [ ]:
# PROJ must be configured before rasterio/localtileserver are imported.
import os
os.environ["PROJ_IGNORE_CELESTIAL_BODY"] = "YES"

from pathlib import Path
import sys
import ipysheet
from IPython.display import Markdown, display
import ipywidgets
import leafmap
import numpy
import pandas
import rasterio
from rasterio.windows import Window
from localtileserver import TileClient, get_leaflet_tile_layer
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from ipyleaflet import WidgetControl

repo_root = Path.cwd().parent
repo_root_str = str(repo_root).replace('/panfs/ccds02/nobackup', '/explore/nobackup')
repo_root = Path(repo_root_str)
NOTEBOOK_DIR = repo_root / "notebooks"

if not (repo_root / "lfm").exists():
  raise FileNotFoundError(
      "Cannot find lfm/ directory. Run this notebook from "
      "lfm/notebooks/full_model or update repo_root."
  )

sys.path.insert(0, str(repo_root))

from model.clustering.Clusterer import Clusterer
from model.clustering.ImageHelperSingleBand import ImageHelper
from model.clustering.ClusterPreprocessConfig import ClusterPreprocessConfig
from model.clustering.clustering_display_utils import display_images_labels, display_images_binary_labels
from model.clustering.clustering_backend_utils import create_cluster_assignment_widget, get_binary_cluster_mapping, summarize_binary_cluster_mapping, crop_center, relabel

# Configuration

`inFile`: Input single-band NAC file to perform clustering on.

`noDataValue`: Nodata value to ignore in clustering. This is vital for the clustering algorithm to properly capture the valid data distribution.

`numClusters`: Number of K-means clusters to create. Higher values create more candidate groups for manual relabeling, but can also make the initial label map noisier.

`cropSize`: Target size for a centered square crop used for clustering and display.

## Preprocess Configuration

`clusterPreprocessConfig` controls optional preprocessing and feature creation before K-means. Leave an option disabled when you want the clustering to use less memory or stay closer to raw brightness values.

`clipPercentiles`: Clips valid pixels to robust lower/upper percentiles before clustering. This reduces the influence of rare extreme values. For this raster, `(1.0, 99.5)` expands contrast in the common terrain range better than min/max scaling.

`gaussianSigma`: Smooths the input before feature creation. Small values such as `1.0` reduce speckle while preserving crater-scale structure. Set to `None` or `0` to disable smoothing.

`includeRaw`: Includes the preprocessed intensity value as a K-means feature. This should usually stay enabled for single-band clustering.

`includeGradientMagnitude`: Adds an edge/texture feature based on local intensity change. This can help separate crater rims and rough terrain from smooth areas with similar brightness.

`includeLocalMean` and `localMeanSize`: Add neighborhood average brightness as a feature. This can make clusters more spatially coherent, but it adds memory and can blur small details.

`includeLocalStd` and `localStdSize`: Add local texture/roughness as a feature. This can help distinguish rough crater interiors or ejecta from smooth background terrain.

`includeLaplacian`: Adds a second-derivative edge feature. This can emphasize rims and sharp transitions, but may be noisy unless smoothing is enabled.

`standardizeFeatures`: Standardizes each enabled feature over valid pixels before K-means. Keep this enabled when using multiple features so one feature does not dominate because of its numeric scale.

`medianFilterLabelsSize`: Optionally applies a median filter to the final cluster labels to reduce salt-and-pepper noise. Use odd values like `3`; set to `None` to disable.

In [ ]:
# Original full-resolution lunar raster.
inFile = "/explore/nobackup/projects/lfm/Benchmarks/Craters/NAC_PHO_E064S3160/NAC_DTM_NEWCRATER6_M1219245090_80CM.TIF"

# Outputs are written here.
outDirectory = ""

# This raster uses 0 as the fill/nodata value around the valid strip.
noDataValue = 0.0

# Number of K-Means clusters to use.
numClusters = 20

# Centered crop size used for this debugging workflow.
cropSize = 5000

clusterPreprocessConfig = ClusterPreprocessConfig(
    clipPercentiles=(1.0, 99.5),
    gaussianSigma=1.0,
    includeRaw=True,
    includeGradientMagnitude=True,
    standardizeFeatures=True,
)

## Path setup

In [ ]:
inFile = Path(inFile)

outDirectory = Path(outDirectory)
outDirectory.mkdir(parents=True, exist_ok=True)

clippedInputFile = outDirectory / f"{inFile.stem}-clip-{cropSize}{inFile.suffix}"
labelsFile = outDirectory / f"{inFile.stem}-clip-{cropSize}-labels{inFile.suffix}"
clusterMapFile = outDirectory / f"{inFile.stem}-clip-{cropSize}-cluster-map{inFile.suffix}"

# Step 1: Clip the input raster

In [ ]:
crop_center(
    src_path=inFile,
    dst_path=clippedInputFile,
    size=cropSize,
)

print(f"Raw input:       {inFile}")
print(f"Clipped input:   {clippedInputFile}")
print(f"Cluster labels:  {labelsFile}")
print(f"Final label map: {clusterMapFile}")

# Step 2: Ingest the clipped raster

In [ ]:
inHelper = ImageHelper()
inHelper.initFromFile(
    inputFile=clippedInputFile,
    noDataValue=noDataValue,
)

print(f"Clustering input shape: {inHelper.getBand().shape}")

# Step 3: Generate first-pass clusters on the clipped raster

These clusters will have many different groupings, equal to numClusters. The second pass will narrow them down to only 2 classes (no crater, crater).

In [ ]:
# Add singleton dimension for the single input band: (H, W) -> (1, H, W).
labels = Clusterer.getClusters(
    bands=numpy.expand_dims(inHelper.getBand(), axis=0),
    numClusters=numClusters,
    noDataValue=noDataValue,
    preprocessConfig=clusterPreprocessConfig,
)

# Because inHelper was created from clippedInputFile, the label GeoTIFF is
# written with the same clipped extent/transform/CRS.
labelsDs = Clusterer.labelsToGeotiff(
    inHelper._dataset,
    labelsFile,
    labels,
)

lHelper = ImageHelper()
lHelper.initFromDataset(labelsDs, noDataValue)

print(f"Labels file written: {labelsFile}")

# Step 4: Display clipped image + clipped labels

In [ ]:
m, legend_control = display_images_labels(clippedInputFile, labelsFile, labels, inHelper, lHelper)
display(m)

## Assign clusters to final classes

Use the cluster color swatches to match each row to the map. Assign crater clusters first, then assign non-crater clusters. The output label convention is fixed: non-crater is class `0`, crater is class `1`, and ignored clusters default to non-crater in the generated binary map.

In [ ]:
clusterAssignmentWidget, clusterAssignmentControls = create_cluster_assignment_widget(labels)
display(clusterAssignmentWidget)

## Build binary labels

Run this cell after assigning clusters above. It will not fail if selections are incomplete; it prints what is currently selected so you can adjust the assignment widget and rerun.

In [ ]:
finalClusters = get_binary_cluster_mapping(clusterAssignmentControls)
summarize_binary_cluster_mapping(finalClusters, labels)

newClusters = relabel(labels, finalClusters)

## Review the updated map

In [ ]:
m = display_images_binary_labels(
    m,
    inHelper,
    clusterMapFile,
    labelsFile,
    newClusters,
    legend_control=legend_control,
)
display(m)